# BitNet b1.58：手写 BitLinear、真实 QAT 与量化失败修复

**面试问题：三值权重、int8 激活、STE、整数点积和真实存储账本怎样连成一个可训练实现？**

## 回答主线

BitNet 的核心不是把权重打印成 -1、0、1，而是保留 FP32 master weight 训练，并在 forward 中用 absmean scale 得到三值码。激活可以按请求动态缩放到 int8，再用量化值参与前向；Straight-Through Estimator 让反向梯度仍回到 master weight。整数码点积乘回两侧 scale 后，应与反量化矩阵乘法数值一致，这是 kernel 实现的重要 oracle。质量比较必须在同一批样本上同时给出 FP32 基线、量化后逐样本预测、loss、梯度和误差。存储账本还要把三值 payload、scale 和 bias 分开，1.58 bit 只描述三值权重的信息下界，不等于整个模型显存。下面不用现成量化框架或 BitNet 实现，只用 Parameter、基础张量运算和手写训练循环完成这些步骤。

## 真实案例：支付风控路由

案例包含 12 条脱敏支付请求，每条记录有登录频次、金额强度、设备风险和地区漂移四个归一化字段，目标动作是正常放行、人工复核或拒绝。低、中、高风险各四条，既能展示逐请求决策，也能观察量化误差是否跨过业务边界。数据是结构与风控特征表一致的离线教学样本，不含真实账号、金额或用户标识；同一小数据上的训练准确率只用于检查计算图，不能外推为线上风控效果。

### 输入预览：12 条请求与四个真实字段

In [1]:
import math  # 导入三值理论位数计算需要的数学函数。
import warnings  # 导入告警控制工具以屏蔽当前教学环境的第三方依赖提示。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 只过滤 PyTorch 导入时由环境包触发的已知提示。
import torch  # 导入 PyTorch 张量、模块和自动微分能力。
torch.set_num_threads(1)  # 固定单线程执行以减少小实验的调度噪声。
torch.manual_seed(2604)  # 固定模型初始化以保证训练输出可复现。
feature_names = ["登录频次", "金额强度", "设备风险", "地区漂移"]  # 定义四个归一化风控字段。
class_names = ["正常放行", "人工复核", "拒绝"]  # 定义三种支付路由动作。
records = [  # 构造十二条有业务语义的脱敏支付请求。
    {"id": "PAY-01", "summary": "低频小额可信设备", "features": [0.10, 0.10, 0.10, 0.10], "label": 0},  # 定义第一条低风险请求。
    {"id": "PAY-02", "summary": "常用设备小额支付", "features": [0.20, 0.20, 0.10, 0.20], "label": 0},  # 定义第二条低风险请求。
    {"id": "PAY-03", "summary": "低频中小额本地支付", "features": [0.10, 0.30, 0.20, 0.10], "label": 0},  # 定义第三条低风险请求。
    {"id": "PAY-04", "summary": "轻微频次升高但设备可信", "features": [0.30, 0.10, 0.20, 0.20], "label": 0},  # 定义第四条低风险请求。
    {"id": "PAY-05", "summary": "频次金额同时升高", "features": [0.60, 0.60, 0.40, 0.30], "label": 1},  # 定义第一条中风险复核请求。
    {"id": "PAY-06", "summary": "大额且地区明显漂移", "features": [0.40, 0.80, 0.50, 0.60], "label": 1},  # 定义第二条中风险复核请求。
    {"id": "PAY-07", "summary": "频次偏高且设备可疑", "features": [0.55, 0.40, 0.45, 0.40], "label": 1},  # 定义第三条中风险复核请求。
    {"id": "PAY-08", "summary": "中大额跨地区支付", "features": [0.50, 0.70, 0.30, 0.70], "label": 1},  # 定义第四条中风险复核请求。
    {"id": "PAY-09", "summary": "高频大额高危设备", "features": [0.90, 0.80, 0.90, 0.80], "label": 2},  # 定义第一条高风险拒绝请求。
    {"id": "PAY-10", "summary": "大额异地高危支付", "features": [0.80, 0.90, 0.80, 0.70], "label": 2},  # 定义第二条高风险拒绝请求。
    {"id": "PAY-11", "summary": "高频设备接管迹象", "features": [0.90, 0.60, 0.95, 0.90], "label": 2},  # 定义第三条高风险拒绝请求。
    {"id": "PAY-12", "summary": "超高金额与地区漂移", "features": [0.70, 0.95, 0.90, 0.95], "label": 2},  # 定义第四条高风险拒绝请求。
]  # 完成十二条支付风控样本。
inputs = torch.tensor([record["features"] for record in records], dtype=torch.float32)  # 把可读记录转换成模型输入矩阵。
labels = torch.tensor([record["label"] for record in records], dtype=torch.long)  # 把目标动作转换成类别索引。
print("输入形状：", tuple(inputs.shape), "字段：", feature_names)  # 展示张量合同和字段顺序。
print("请求 | 场景摘要 | 四维特征 | 目标动作")  # 输出逐请求输入表头。
for record in records:  # 逐条展示十二条业务输入。
    print(f"{record['id']} | {record['summary']} | {record['features']} | {class_names[record['label']]}")  # 输出可读特征和人工目标。

输入形状： (12, 4) 字段： ['登录频次', '金额强度', '设备风险', '地区漂移']
请求 | 场景摘要 | 四维特征 | 目标动作
PAY-01 | 低频小额可信设备 | [0.1, 0.1, 0.1, 0.1] | 正常放行
PAY-02 | 常用设备小额支付 | [0.2, 0.2, 0.1, 0.2] | 正常放行
PAY-03 | 低频中小额本地支付 | [0.1, 0.3, 0.2, 0.1] | 正常放行
PAY-04 | 轻微频次升高但设备可信 | [0.3, 0.1, 0.2, 0.2] | 正常放行
PAY-05 | 频次金额同时升高 | [0.6, 0.6, 0.4, 0.3] | 人工复核
PAY-06 | 大额且地区明显漂移 | [0.4, 0.8, 0.5, 0.6] | 人工复核
PAY-07 | 频次偏高且设备可疑 | [0.55, 0.4, 0.45, 0.4] | 人工复核
PAY-08 | 中大额跨地区支付 | [0.5, 0.7, 0.3, 0.7] | 人工复核
PAY-09 | 高频大额高危设备 | [0.9, 0.8, 0.9, 0.8] | 拒绝
PAY-10 | 大额异地高危支付 | [0.8, 0.9, 0.8, 0.7] | 拒绝
PAY-11 | 高频设备接管迹象 | [0.9, 0.6, 0.95, 0.9] | 拒绝
PAY-12 | 超高金额与地区漂移 | [0.7, 0.95, 0.9, 0.95] | 拒绝


## Baseline 基线：FP32 线性层

先用同一组 Parameter、矩阵乘法和手写交叉熵训练 FP32 线性分类器。它是 BitLinear 的质量 reference：量化方案必须面对完全相同的 12 条请求和标签。训练循环不用 Trainer 或现成优化器，而是显式执行 forward、backward、梯度下降和清零。

In [2]:
class FullPrecisionLinear(torch.nn.Module):  # 定义最小 FP32 线性分类基线。
    def __init__(self, input_dim=4, output_dim=3):  # 初始化输入和输出维度。
        super().__init__()  # 注册 PyTorch 模块参数管理能力。
        self.weight = torch.nn.Parameter(torch.randn(output_dim, input_dim) * 0.10)  # 初始化三乘四的 FP32 权重。
        self.bias = torch.nn.Parameter(torch.zeros(output_dim))  # 初始化三个类别偏置。
    def forward(self, batch_inputs):  # 实现基础线性层的真实前向传播。
        return batch_inputs @ self.weight.t() + self.bias  # 返回十二条请求的三类 logits。
def manual_cross_entropy(logits, target_labels):  # 手写数值稳定的多分类交叉熵。
    log_probabilities = logits - torch.logsumexp(logits, dim=1, keepdim=True)  # 通过 log-sum-exp 计算稳定对数概率。
    row_indices = torch.arange(target_labels.shape[0])  # 构造逐样本行索引。
    return -log_probabilities[row_indices, target_labels].mean()  # 取目标类别负对数概率并求平均。
fp_model = FullPrecisionLinear()  # 实例化 FP32 reference 模型。
fp_learning_rate = 0.12  # 设置全批量 FP32 训练学习率。
fp_trace = []  # 保存 FP32 训练过程的关键快照。
for step in range(501):  # 执行五百次真实 FP32 训练更新。
    fp_logits = fp_model(inputs)  # 运行 FP32 线性层 forward。
    fp_loss = manual_cross_entropy(fp_logits, labels)  # 计算当前参数的真实分类损失。
    fp_loss.backward()  # 将损失梯度传播到 FP32 权重与偏置。
    if step in {0, 100, 300, 500}:  # 在固定训练步记录指标。
        fp_accuracy = float((fp_logits.argmax(dim=1) == labels).float().mean())  # 计算同一批样本上的基线准确率。
        fp_trace.append((step, float(fp_loss.detach()), fp_accuracy))  # 保存损失与准确率用于展示。
    with torch.no_grad():  # 关闭参数更新阶段的梯度记录。
        for parameter in fp_model.parameters():  # 遍历 FP32 模型的权重和偏置。
            parameter -= fp_learning_rate * parameter.grad  # 使用基础梯度下降更新参数。
            parameter.grad.zero_()  # 清空梯度避免下一步错误累加。
with torch.no_grad():  # 在训练结束后关闭梯度计算。
    fp_logits = fp_model(inputs)  # 重新计算最终 FP32 logits。
    fp_predictions = fp_logits.argmax(dim=1)  # 得到最终 FP32 类别预测。
    fp_accuracy = float((fp_predictions == labels).float().mean())  # 计算最终基线准确率。
print("step | fp32_loss | fp32_accuracy")  # 输出 FP32 训练曲线表头。
for step, loss_value, accuracy in fp_trace:  # 逐个展示训练快照。
    print(f"{step:4d} | {loss_value:9.4f} | {accuracy:6.1%}")  # 展示真实损失下降和准确率变化。
print(f"FP32 最终准确率：{fp_accuracy:.1%}")  # 输出后续量化比较使用的质量 reference。

step | fp32_loss | fp32_accuracy
   0 |    1.1134 |  33.3%
 100 |    0.7118 |  83.3%
 300 |    0.4633 | 100.0%
 500 |    0.3564 | 100.0%
FP32 最终准确率：100.0%


## 核心实现：手写 BitLinear forward

权重路径先用 absmean 计算一个 scale，再把 FP32 master weight 舍入到 -1、0、1；前向使用三值重建权重，反向通过 STE 把梯度送回 master weight。激活路径按每条请求的最大绝对值计算 int8 scale，避免一个异常请求污染整批。audit 字典保存真实整数码、scale 和整数点积，便于验证数学等价与部署制品。

In [3]:
class BitLinear(torch.nn.Module):  # 定义可训练的三值权重线性层。
    def __init__(self, input_dim=4, output_dim=3):  # 初始化输入维度与动作类别数。
        super().__init__()  # 注册 PyTorch 模块参数管理能力。
        self.master_weight = torch.nn.Parameter(torch.randn(output_dim, input_dim) * 0.10)  # 保存训练阶段的 FP32 主权重。
        self.bias = torch.nn.Parameter(torch.zeros(output_dim))  # 保存不做三值化的 FP32 偏置。
    def forward(self, batch_inputs):  # 实现三值权重和 int8 激活的真实前向传播。
        weight_scale = self.master_weight.detach().abs().mean().clamp_min(1e-8)  # 用全权重 absmean 计算三值缩放因子。
        weight_codes = torch.round(self.master_weight / weight_scale).clamp(-1, 1)  # 把主权重舍入到负一、零、正一。
        quantized_weight = self.master_weight + (weight_codes * weight_scale - self.master_weight).detach()  # 用 STE 让前向取三值重建而反向对主权重近似恒等。
        activation_scale = batch_inputs.detach().abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 127.0  # 为每条请求计算独立 int8 scale。
        activation_codes = torch.round(batch_inputs / activation_scale).clamp(-127, 127)  # 把每条请求量化到对称 int8 范围。
        quantized_inputs = batch_inputs + (activation_codes * activation_scale - batch_inputs).detach()  # 用 STE 让前向使用反量化激活。
        logits = quantized_inputs @ quantized_weight.t() + self.bias  # 计算量化路径的三类 logits。
        integer_dot = activation_codes @ weight_codes.t()  # 计算不含 scale 与 bias 的整数码点积。
        audit = {  # 收集解释量化路径需要的中间状态。
            "weight_scale": weight_scale,  # 保存三值权重 scale。
            "weight_codes": weight_codes,  # 保存负一、零、正一权重码。
            "activation_scale": activation_scale,  # 保存逐请求激活 scale。
            "activation_codes": activation_codes,  # 保存逐请求 int8 激活码。
            "integer_dot": integer_dot,  # 保存整数点积结果。
        }  # 完成量化审计账本。
        return logits, audit  # 返回可训练 logits 与可验证中间状态。
bit_model = BitLinear()  # 实例化手写 BitLinear 模型。
with torch.no_grad():  # 关闭从 FP32 reference 拷贝参数时的梯度记录。
    bit_model.master_weight.copy_(fp_model.weight)  # 用已训练 FP32 权重初始化 QAT 主权重。
    bit_model.bias.copy_(fp_model.bias)  # 同步 FP32 偏置以建立公平起点。
initial_bit_logits, initial_audit = bit_model(inputs)  # 运行一次真实量化前向获得整数码。
print("初始三值权重码：")  # 输出权重码矩阵标题。
print(initial_audit["weight_codes"].to(torch.int8))  # 展示每个权重确实落在三值集合。
print("前两条请求的 int8 激活码：")  # 输出激活量化示例标题。
print(initial_audit["activation_codes"][:2].to(torch.int16))  # 展示逐请求动态量化后的整数输入。

初始三值权重码：
tensor([[-1, -1, -1, -1],
        [ 0,  1, -1,  0],
        [ 1,  0,  1,  1]], dtype=torch.int8)
前两条请求的 int8 激活码：
tensor([[127, 127, 127, 127],
        [127, 127,  64, 127]], dtype=torch.int16)


### 真实 QAT：量化 forward、backward 与 STE 梯度

下面在量化路径上继续训练 master weight。每一步的 logits 都来自三值权重和 int8 激活，而 backward 通过 STE 更新 FP32 参数。训练完成后额外执行一次梯度探针，并把整数点积乘回 scale，与反量化矩阵乘法逐值比较。

In [4]:
bit_learning_rate = 0.04  # 设置量化感知微调的基础学习率。
bit_trace = []  # 保存 QAT 过程的关键损失与准确率。
for step in range(601):  # 执行六百次真实量化前向和反向更新。
    bit_logits, bit_audit = bit_model(inputs)  # 运行三值权重与 int8 激活的 forward。
    bit_loss = manual_cross_entropy(bit_logits, labels)  # 在量化 logits 上计算真实分类损失。
    bit_loss.backward()  # 通过 STE 把梯度传回 FP32 master weight。
    if step in {0, 100, 300, 600}:  # 在固定步数记录 QAT 指标。
        bit_accuracy = float((bit_logits.argmax(dim=1) == labels).float().mean())  # 计算量化路径的逐样本准确率。
        bit_trace.append((step, float(bit_loss.detach()), bit_accuracy))  # 保存损失与准确率快照。
    with torch.no_grad():  # 关闭参数更新阶段的梯度记录。
        for parameter in bit_model.parameters():  # 遍历 master weight 与 bias。
            parameter -= bit_learning_rate * parameter.grad  # 使用基础梯度下降更新 FP32 参数。
            parameter.grad.zero_()  # 清空梯度避免下一步错误累加。
bit_model.zero_grad()  # 清空训练结束后的残余梯度以准备独立探针。
probe_logits, probe_audit = bit_model(inputs)  # 重新执行量化 forward 构造独立梯度探针。
probe_loss = manual_cross_entropy(probe_logits, labels)  # 计算探针损失。
probe_loss.backward()  # 真实执行一次 backward 验证 STE 梯度。
gradient_norm = float(bit_model.master_weight.grad.norm())  # 读取 FP32 主权重收到的梯度范数。
integer_restored = probe_audit["integer_dot"] * probe_audit["activation_scale"] * probe_audit["weight_scale"]  # 把整数点积乘回激活和权重 scale。
matrix_restored = (probe_audit["activation_codes"] * probe_audit["activation_scale"]) @ (probe_audit["weight_codes"] * probe_audit["weight_scale"]).t()  # 用反量化张量执行 reference 矩阵乘法。
integer_equivalence_error = float((integer_restored - matrix_restored).abs().max())  # 计算整数路径与 reference 的最大误差。
print("step | bit_loss | bit_accuracy")  # 输出 QAT 训练曲线表头。
for step, loss_value, accuracy in bit_trace:  # 逐个展示量化训练快照。
    print(f"{step:4d} | {loss_value:8.4f} | {accuracy:6.1%}")  # 展示量化 forward 下的真实训练变化。
print(f"STE 主权重梯度范数：{gradient_norm:.6f}")  # 证明 backward 确实到达 FP32 master weight。
print(f"整数点积恢复最大误差：{integer_equivalence_error:.8f}")  # 展示整数 kernel oracle 的数值等价性。

step | bit_loss | bit_accuracy
   0 |   0.5375 |  75.0%
 100 |   0.4200 | 100.0%
 300 |   0.3801 | 100.0%
 600 |   0.3452 | 100.0%
STE 主权重梯度范数：0.058322
整数点积恢复最大误差：0.00000048


## 结果解读：逐请求质量、量化误差与存储账本

质量不能只看一个 assert。下面逐条输出人工目标、FP32 预测、BitLinear 预测和两组 logits 的最大差异，同时计算总体准确率。存储比较把三值 payload、一个权重 scale 和 FP32 bias 都计入；它仍是单层教学下界，不包含优化器状态、激活、打包对齐或 kernel workspace。

In [5]:
with torch.no_grad():  # 在最终结果统计阶段关闭梯度。
    final_fp_logits = fp_model(inputs)  # 计算同一批请求的 FP32 reference logits。
    final_bit_logits, final_audit = bit_model(inputs)  # 计算同一批请求的三值量化 logits。
    final_fp_predictions = final_fp_logits.argmax(dim=1)  # 得到 FP32 逐请求决策。
    final_bit_predictions = final_bit_logits.argmax(dim=1)  # 得到 BitLinear 逐请求决策。
    final_bit_accuracy = float((final_bit_predictions == labels).float().mean())  # 计算量化模型准确率。
    logits_max_error = float((final_bit_logits - final_fp_logits).abs().max())  # 计算量化与 FP32 logits 最大差异。
weight_count = bit_model.master_weight.numel()  # 统计三值权重元素数量。
bias_count = bit_model.bias.numel()  # 统计 FP32 bias 元素数量。
fp32_storage_bits = (weight_count + bias_count) * 32  # 计算 FP32 权重与偏置的理论位数。
ternary_payload_bits = math.ceil(weight_count * math.log2(3))  # 计算三值码的 base-3 信息下界。
quantized_storage_bits = ternary_payload_bits + 32 + bias_count * 32  # 加上一个 FP32 权重 scale 和 FP32 bias。
compression_ratio = fp32_storage_bits / quantized_storage_bits  # 计算教学制品相对 FP32 的理论压缩比。
print("请求 | 目标 | FP32 | BitLinear | 最大logit差")  # 输出逐请求结果表头。
for index, record in enumerate(records):  # 遍历十二条支付请求。
    row_error = float((final_bit_logits[index] - final_fp_logits[index]).abs().max())  # 计算当前请求的最大 logit 偏差。
    print(f"{record['id']} | {class_names[labels[index]]} | {class_names[final_fp_predictions[index]]} | {class_names[final_bit_predictions[index]]} | {row_error:.4f}")  # 展示同一请求上的真实对照。
print(f"准确率：FP32={fp_accuracy:.1%}，BitLinear={final_bit_accuracy:.1%}，全局最大logit差={logits_max_error:.4f}")  # 汇总质量与数值误差。
print(f"理论存储：FP32={fp32_storage_bits} bit，三值制品={quantized_storage_bits} bit，压缩={compression_ratio:.2f}x")  # 展示包含 scale 与 bias 的透明存储账本。

请求 | 目标 | FP32 | BitLinear | 最大logit差
PAY-01 | 正常放行 | 正常放行 | 正常放行 | 0.1512
PAY-02 | 正常放行 | 正常放行 | 正常放行 | 0.1198
PAY-03 | 正常放行 | 正常放行 | 正常放行 | 0.2869
PAY-04 | 正常放行 | 正常放行 | 正常放行 | 0.1122
PAY-05 | 人工复核 | 人工复核 | 人工复核 | 0.2865
PAY-06 | 人工复核 | 人工复核 | 人工复核 | 0.3790
PAY-07 | 人工复核 | 人工复核 | 人工复核 | 0.2303
PAY-08 | 人工复核 | 人工复核 | 人工复核 | 0.1537
PAY-09 | 拒绝 | 拒绝 | 拒绝 | 0.4670
PAY-10 | 拒绝 | 拒绝 | 拒绝 | 0.4340
PAY-11 | 拒绝 | 拒绝 | 拒绝 | 0.6326
PAY-12 | 拒绝 | 拒绝 | 拒绝 | 0.4555
准确率：FP32=100.0%，BitLinear=100.0%，全局最大logit差=0.6326
理论存储：FP32=480 bit，三值制品=148 bit，压缩=3.24x


## 失败案例与修正：整批共享激活 scale 被异常单位污染

假设 PAY-12 的金额字段漏做归一化，从 0.95 变成 100。错误实现给整个 batch 共用一个 int8 scale，导致其余正常请求的大量小特征被舍入为零。修正分两层：量化采用逐请求 scale 隔离数值范围；输入合同再用 0–1 范围门禁拒绝单位异常，不能靠量化器替数据管线兜底。

In [6]:
outlier_inputs = inputs.clone()  # 复制正常批次以构造单位异常反例。
outlier_inputs[-1, 1] = 100.0  # 模拟 PAY-12 金额忘记归一化而放大约一百倍。
global_scale = outlier_inputs.abs().max().clamp_min(1e-8) / 127.0  # 错误地为整个批次只计算一个激活 scale。
global_codes = torch.round(outlier_inputs / global_scale).clamp(-127, 127)  # 使用异常值主导的 scale 量化全部请求。
global_restored = global_codes * global_scale  # 反量化错误的整批共享 scale 结果。
row_scales = outlier_inputs.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 127.0  # 为每条请求独立计算修正后的 scale。
row_codes = torch.round(outlier_inputs / row_scales).clamp(-127, 127)  # 使用逐请求 scale 量化各自特征。
row_restored = row_codes * row_scales  # 反量化隔离异常范围后的请求特征。
normal_global_mse = float(((global_restored[:-1] - outlier_inputs[:-1]) ** 2).mean())  # 计算错误方案对十一条正常请求的重建误差。
normal_row_mse = float(((row_restored[:-1] - outlier_inputs[:-1]) ** 2).mean())  # 计算逐请求方案对正常请求的重建误差。
global_zero_rate = float((global_codes[:-1] == 0).float().mean())  # 统计错误方案把正常特征压成零的比例。
row_zero_rate = float((row_codes[:-1] == 0).float().mean())  # 统计逐请求方案的零码比例。
range_mask = (outlier_inputs >= 0.0).all(dim=1) & (outlier_inputs <= 1.0).all(dim=1)  # 根据输入合同检查每条请求是否都在零到一范围。
rejected_ids = [records[index]["id"] for index in range(len(records)) if not range_mask[index]]  # 收集被范围门禁拒绝的单位异常请求。
print(f"整批共享 scale={global_scale:.4f}，正常请求 MSE={normal_global_mse:.6f}，零码率={global_zero_rate:.1%}")  # 展示异常值如何污染其他请求。
print(f"逐请求 scale 后正常请求 MSE={normal_row_mse:.6f}，零码率={row_zero_rate:.1%}")  # 展示数值隔离后的真实改进。
print("输入范围门禁拒绝：", rejected_ids)  # 展示修正方案不会静默处理单位错误。
print("修正：per-row 激活量化负责隔离范围，schema 门禁负责阻断错误单位。")  # 区分数值算法与数据治理的职责。

整批共享 scale=0.7874，正常请求 MSE=0.042494，零码率=40.9%
逐请求 scale 后正常请求 MSE=0.000001，零码率=0.0%
输入范围门禁拒绝： ['PAY-12']
修正：per-row 激活量化负责隔离范围，schema 门禁负责阻断错误单位。


### 生产差距

本例只有 12 个三值权重，整数点积仍由 PyTorch 浮点张量模拟；真实加速依赖紧凑 base-3 或二位打包、专用矩阵 kernel、对齐布局和端到端 profiling。训练还会保留 FP32 master weight、梯度和优化器状态，因此训练显存不会按 1.58 bit 缩小。生产发布必须记录量化规则、scale 粒度、校准数据版本、输入 schema、bias dtype 和 fallback 策略。

In [7]:
bitnet_artifact = {  # 构造可审计的 BitLinear 发布元数据。
    "weight_codes": "ternary_-1_0_1",  # 声明权重 payload 的三值编码。
    "weight_scale": "absmean_per_tensor",  # 声明当前示例的权重 scale 粒度。
    "activation_codes": "int8_symmetric_per_row",  # 声明激活按请求独立量化。
    "master_weight": "fp32_training_only",  # 声明 FP32 主权重只属于训练制品。
    "input_contract": "four_features_in_0_1",  # 声明进入量化层前必须满足的字段范围。
    "kernel": "teaching_integer_oracle",  # 声明当前实现是教学 reference 而非高性能 kernel。
}  # 完成训练与服务都需要读取的量化合同。
print("BitLinear 发布制品：", bitnet_artifact)  # 展示不能只保存一个“1.58bit”标签。
print("最终三值码分布：", {int(code): int((final_audit["weight_codes"] == code).sum()) for code in (-1, 0, 1)})  # 展示三值权重是否实际使用多个码。
print("教学实验边界：准确率与压缩比只对应本例单层和 12 条训练样本。")  # 防止把受控结果冒充基础模型收益。

BitLinear 发布制品： {'weight_codes': 'ternary_-1_0_1', 'weight_scale': 'absmean_per_tensor', 'activation_codes': 'int8_symmetric_per_row', 'master_weight': 'fp32_training_only', 'input_contract': 'four_features_in_0_1', 'kernel': 'teaching_integer_oracle'}
最终三值码分布： {-1: 5, 0: 3, 1: 4}
教学实验边界：准确率与压缩比只对应本例单层和 12 条训练样本。


## 最小回归测试

断言只保护三值范围、真实 STE 梯度、整数路径等价、样本量、量化决策和失败修复；主要学习证据是前面的训练曲线、逐请求表和单位污染对照。

In [8]:
assert len(records) >= 5  # 保证支付案例至少包含五条有业务语义的请求。
assert set(final_audit["weight_codes"].unique().tolist()) <= {-1.0, 0.0, 1.0}  # 保证权重码严格属于三值集合。
assert gradient_norm > 0.0 and torch.isfinite(bit_model.master_weight.grad).all()  # 保证 STE 为 FP32 主权重提供有限非零梯度。
assert integer_equivalence_error < 1e-5  # 保证整数码点积乘回 scale 后等价于反量化矩阵乘法。
assert final_bit_accuracy == 1.0  # 保证量化模型在本教学集上逐条保持人工目标。
assert normal_row_mse < normal_global_mse * 0.01 and rejected_ids == ["PAY-12"]  # 保证逐请求 scale 与范围门禁真实修复单位污染。
print("回归测试通过：真实样本、三值 forward、STE backward、整数 oracle、逐条决策和失败修复均成立。")  # 汇总少量关键不变量。

回归测试通过：真实样本、三值 forward、STE backward、整数 oracle、逐条决策和失败修复均成立。
